II. Quantum ESPRESSO

Quantum ESPRESSO (QE) is an open‑source suite of plane‑wave Density Functional Theory (DFT) codes designed for electronic‑structure calculations and materials modeling. Unlike LCAO‑based codes such as SIESTA, QE uses plane‑wave basis sets together with pseudopotentials, making it particularly well suited for periodic systems, crystalline materials, and simulations requiring systematic convergence with respect to basis‑set size.

Because plane waves form a complete and unbiased basis, QE offers high numerical accuracy and predictable convergence behavior. This makes it a popular choice for studying band structures, phonons, dielectric properties, and many other phenomena where precision is essential. QE also supports a wide range of pseudopotential formats, including norm‑conserving, ultrasoft, and PAW datasets, giving users flexibility in balancing accuracy and computational cost.

The code is parallelized efficiently using MPI and OpenMP, enabling large‑scale simulations on modern HPC systems. Its active development community continuously expands its capabilities, integrates new methods, and maintains a rich ecosystem of tools for pre‑ and post‑processing.

Overall, Quantum ESPRESSO provides a robust, versatile, and widely adopted platform for first‑principles materials modeling, complementing LCAO‑based codes and offering a different set of strengths for high‑accuracy plane‑wave calculations.

1. A single‑point calculation performed with Quantum ESPRESSO introduces the essential structure of a QE workflow: defining the atomic system, selecting the plane‑wave cutoff energies and pseudopotentials, configuring the pw.x input parameters, and executing the self‑consistent field (SCF) cycle to obtain the electronic ground‑state properties without performing any geometry optimization.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.build import bulk
from ase.visualize import view
profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)


os.mkdir("c2_qe")
os.chdir("c2_qe")

a = 3.567  # lattice constant in Å

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

# Minimal Quantum Espresso calculator for testing
atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe'        # ← THIS is the QE system label
            },
            'system': {
                'ecutwfc': 30,
                'ecutrho': 105
            },
            'electrons': {
                'conv_thr': 1e-8
            }
        },
        kpts=(4, 4, 4)
    )

energy = atoms.get_total_energy()

# Run a single-point calculation

os.chdir("../")

print("Energy:", energy)

view(atoms, viewer='x3d')

2. Converging the k‑point mesh in Quantum ESPRESSO is essential for obtaining reliable and reproducible plane‑wave DFT results. Because QE samples the Brillouin zone using a discrete grid, the accuracy of total energies, forces, and electronic properties depends directly on the density of this k‑point sampling. The required mesh is determined by the Bravais lattice and the reciprocal‑space dimensions of the periodic cell: large real‑space cells require fewer k‑points, while compact cells demand a denser grid. As in SIESTA, the optimal k‑point mesh is unique to each system and cannot be transferred from another calculation. The standard procedure is to systematically increase the k‑point density and monitor the convergence of the total energy (and, when needed, forces or stress). Only once the energy changes fall below a chosen threshold can the mesh be considered converged, ensuring that subsequent geometry optimizations, band‑structure calculations, and DOS analyses are built on a solid numerical foundation.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.build import bulk
import numpy as np
import matplotlib.pyplot as plt
profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

os.mkdir("c2_qe_kpts")
os.chdir("c2_qe_kpts")

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

kk_array = []
energy_array = []


for kk in range(1, 9, 1):
    atoms.calc = Espresso(
            pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
            profile=profile,
            input_data={
                'control': {
                    'calculation': 'scf',
                    'prefix': 'c2_qe'        # ← THIS is the QE system label
                },
                'system': {
                    'ecutwfc': 30,
                    'ecutrho': 105
                },
                'electrons': {
                    'conv_thr': 1e-8
                }
            },
            kpts=(kk, kk, kk)
        )
    energy = atoms.get_total_energy()
    print("k-points:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)

# Run a single-point calculation

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs kk')  # Add a title to the plot
plt.xlabel('kk')  # Label for the x-axis
plt.ylabel('Energy')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")


3. In Quantum ESPRESSO, converging the plane‑wave cutoff energies is essential for obtaining accurate and reproducible results. QE uses two separate cutoffs: ecutwfc for the Kohn–Sham wavefunctions and ecutrho for the charge density. Because the charge density contains products of wavefunctions, it requires a much larger basis set—typically four times the wavefunction cutoff for norm‑conserving pseudopotentials, and 8–12× for ultrasoft or PAW datasets. These cutoffs determine how many plane waves are included in the expansion, directly affecting the precision of total energies, forces, stresses, and electronic properties. The required values depend strongly on the pseudopotential and cannot be transferred from another system. Therefore, the standard procedure is to systematically increase ecutwfc (and the corresponding ecutrho) until the total energy and forces change by less than a chosen threshold. Only after this convergence is achieved can subsequent geometry optimizations, DOS calculations, and phonon simulations be considered numerically reliable.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.build import bulk
import numpy as np
import matplotlib.pyplot as plt
profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

os.mkdir("c2_qe_cutoff")
os.chdir("c2_qe_cutoff")

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

kk_array = []
energy_array = []


for kk in range(30, 95, 5):
    atoms.calc = Espresso(
            pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
            profile=profile,
            input_data={
                'control': {
                    'calculation': 'scf',
                    'prefix': 'c2_qe'        # ← THIS is the QE system label
                },
                'system': {
                    'ecutwfc': kk,
                    'ecutrho': 4*kk
                },
                'electrons': {
                    'conv_thr': 1e-8
                }
            },
            kpts=(5, 5, 5)
        )
    energy = atoms.get_total_energy()
    print("cutoff:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)

# Run a single-point calculation

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs cutoff')  # Add a title to the plot
plt.xlabel('cutoff')  # Label for the x-axis
plt.ylabel('Energy')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")


4. Geometry and lattice optimization in this tutorial are performed using the ASE optimization algorithms, while Quantum ESPRESSO serves strictly as the force and energy calculator. This separation of roles is intentional. Although QE provides built‑in relaxation modes, external frameworks such as ASE offer modern, robust, and flexible optimization algorithms that integrate naturally with Python workflows and often provide smoother convergence for both atomic positions and cell degrees of freedom. The optimizer used here is the Broyden–Fletcher–Goldfarb–Shanno (BFGS) algorithm, a quasi‑Newton method that combines gradient information with an evolving approximation of the inverse Hessian. This approach accelerates convergence by effectively estimating the curvature of the energy landscape without requiring explicit second‑derivative calculations. In contrast, QE’s internal Conjugate Gradient (CG) relaxations rely solely on first‑order information. CG constructs search directions that are conjugate with respect to the Hessian but never stores or approximates the Hessian itself. While efficient and widely used, CG typically converges more slowly and less smoothly than quasi‑Newton methods, especially for complex materials or systems with soft vibrational modes. To enable full lattice optimization, ASE provides the UnitCellFilter class, which exposes the cell degrees of freedom to the optimizer. When wrapped around the atomic configuration, UnitCellFilter ensures that both atomic positions and lattice vectors are updated consistently during the optimization process, allowing QE to supply accurate forces and stresses while ASE handles the optimization logic.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.build import bulk
from ase.visualize import view
from ase.optimize.bfgs import BFGS
from ase.filters import UnitCellFilter

from ase.io import write


profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)


os.mkdir("c2_qe_opt")
os.chdir("c2_qe_opt")

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

print("Initial cell:")
print(atoms.cell)

# Minimal Quantum Espresso calculator for testing
atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80
            },
            'electrons': {
                'conv_thr': 1e-8
            }
        },
        kpts=(5, 5, 5)
    )

energy = atoms.get_total_energy()

ucf = UnitCellFilter(atoms)
opt = BFGS(ucf, trajectory='cellopt.traj')
opt.run(fmax=0.01)

# Run a single-point calculation

os.chdir("../")

print("Final cell:")
print(atoms.cell)

write('cell_qe.traj', atoms)

view(atoms, viewer='x3d')

5. The Density of States (DOS) is one of the fundamental electronic properties computed in Density Functional Theory. It describes how many electronic states are available at each energy level and provides direct insight into the material’s electronic behavior. By examining the DOS, we can identify the valence band, conduction band, and the band gap, which together determine whether a material behaves as an insulator, semiconductor, or metal. Because of this, DOS analysis is an essential step in interpreting and understanding the electronic structure obtained from DFT calculations.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.io import read
from ase.dft.dos import DOS
import matplotlib.pyplot as plt

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

atoms = read('cell_qe.traj')
atoms.pbc=True

os.mkdir("c2_qe_dos")
os.chdir("c2_qe_dos")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
                'nbnd': 48
            },
            'electrons': {
                'conv_thr': 1e-8
            }
        },
        kpts=(5, 5, 5)
    )

energy = atoms.get_potential_energy()

dos = DOS(atoms.calc,
          width=0.05,
          npts=3000)

E = dos.get_energies()
D = dos.get_dos()

os.chdir("../")

# Select only -10 to +10 eV around Ef
mask = (E >= -10) & (E <= 10)

plt.plot(E[mask], D[mask])
plt.axvline(0, color='k', linestyle='--')
plt.xlim(-10, 10)
plt.xlabel(r'$E - E_F$ (eV)')
plt.ylabel('DOS (states/eV)')
plt.show()


6. Surface with QE

In [ ]:
from ase import Atoms
from ase.build import surface
from ase.visualize import view
from ase.io import read
from ase.io import write
import os
from ase.optimize.bfgs import BFGS
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.optimize.bfgs import BFGS

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

slab = read('slab_siesta.traj')
slab.pbc=True

# center slab in vacuum
slab.center(axis=2)

os.mkdir("c2_qe_slab_opt")
os.chdir("c2_qe_slab_opt")

# Minimal Vasp calculator for testing
slab.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
                'assume_isolated':'2D'
            },
            'electrons': {
                'mixing_beta': 0.1,
                'electron_maxstep': 300,
                'conv_thr': 1e-6
            }
        },
        kpts=(5, 5, 1)
    )


energy = slab.get_total_energy()

opt = BFGS(slab, trajectory='cellopt.traj')
opt.run(fmax=0.03)

# Run a single-point calculation

os.chdir("../")

write('slab_qe.traj', slab)

view(slab, viewer='x3d')

7. The surface Density of States (DOS) provides essential insight into how a surface modifies the electronic structure of a material. Unlike the bulk, where all bonds are fully coordinated, a surface exposes atoms with unsatisfied valence—leading to electronic features that can differ dramatically from the bulk behavior. The DOS of the diamond (100) surface illustrates this clearly. Although bulk diamond is a wide‑band‑gap insulator, the (100) surface shows no band gap and exhibits metallic character. At first glance this result may seem counterintuitive. However, the explanation lies in the orbital configuration of the surface atoms. The top‑layer carbon atoms possess dangling bonds that are not compensated by neighboring atoms. These dangling‑bond states fall within the band gap of bulk diamond and create partially filled surface states, giving rise to metallic behavior. This metallicity is not just a computational artifact—it is consistent with experimental observations, where clean diamond surfaces often display surface conductivity due to these unsaturated bonds. Understanding this effect is crucial when studying surface chemistry, adsorption, catalysis, or electronic devices based on diamond surfaces.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.io import read
from ase.dft.dos import DOS
import matplotlib.pyplot as plt

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

atoms = read('slab_qe.traj')
atoms.pbc=True

os.mkdir("c2_qe_slab_dos")
os.chdir("c2_qe_slab_dos")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
                'occupations': 'smearing',
                'smearing': 'mv',
                'degauss': 0.02,
                # no nspin, no starting_magnetization → non‑polarized
                'assume_isolated': '2D',
                'nbnd': 70
                
            },
            'electrons': {
                'conv_thr': 1e-6,
                'electron_maxstep': 500,
                'mixing_mode': 'local-TF',
                'mixing_beta': 0.2,
                'mixing_ndim': 8,
            }
        },
        kpts=(5, 5, 1)
    )

energy = atoms.get_potential_energy()

dos = DOS(atoms.calc,
          width=0.05,
          npts=3000)

E = dos.get_energies()
D = dos.get_dos()

os.chdir("../")

# Select only -10 to +10 eV around Ef
mask = (E >= -10) & (E <= 6)

plt.plot(E[mask], D[mask])
plt.axvline(0, color='k', linestyle='--')
plt.xlim(-10, 6)
plt.xlabel(r'$E - E_F$ (eV)')
plt.ylabel('DOS (states/eV)')
plt.show()


8. To hydrogen‑passivate the surface, we begin by reading the optimized slab obtained from the SIESTA calculations and extract only the positions of the hydrogen atoms. These hydrogen coordinates are then added to the unpassivated slab optimized with Quantum ESPRESSO, ensuring that the passivation pattern is physically meaningful while keeping the underlying lattice consistent with the QE calculation.

In [ ]:
from ase.visualize import view
from ase.io import read
from ase.io import write
import os
from ase.optimize.bfgs import BFGS
from ase.calculators.espresso import Espresso, EspressoProfile

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

atoms2_with_H = read('slab_h_siesta_high.traj')
atoms2_with_H.pbc = True

os.mkdir("c2_qe_h_qe_high_opt")
os.chdir("c2_qe_h_qe_high_opt")

# Minimal Vasp calculator for testing
atoms2_with_H.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF', 'H': 'H.pbe-rrkjus_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80
            },
            'electrons': {
                'mixing_beta': 0.3,
                'electron_maxstep': 300,
                'conv_thr': 1e-6
            }
        },
        kpts=(5, 5, 1)
    )

energy = atoms2_with_H.get_total_energy()

opt = BFGS(atoms2_with_H, trajectory='cellopt.traj')
opt.run(fmax=0.03)

# Run a single-point calculation

os.chdir("../")

write('slab_h_qe_high.traj', atoms2_with_H)

view(atoms2_with_H, viewer='x3d')

9. The Density of States (DOS) of the hydrogen‑passivated diamond (100) surface shows a large band gap, closely matching that of bulk diamond. This behavior is expected: once the surface carbon atoms are saturated with hydrogen, the dangling‑bond states responsible for the metallic character of the pristine surface are removed. With these mid‑gap states eliminated, the electronic structure of the slab returns to that of an insulator, demonstrating that hydrogen termination effectively restores the bulk‑like electronic properties at the surface.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.io import read
from ase.dft.dos import DOS
import matplotlib.pyplot as plt

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

atoms = read('slab_h_qe_high.traj')
atoms.pbc=True

os.mkdir("c2_qe_slab_h_dos_high")
os.chdir("c2_qe_slab_h_dos_high")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF', 'H': 'H.pbe-rrkjus_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
                'nbnd': 70
            },
            'electrons': {
                'conv_thr': 1e-6
            }
        },
        kpts=(5, 5, 1)
    )

energy = atoms.get_potential_energy()

dos = DOS(atoms.calc,
          width=0.05,
          npts=3000)

E = dos.get_energies()
D = dos.get_dos()

os.chdir("../")

# Select only -10 to +10 eV around Ef
mask = (E >= -10) & (E <= 6)

plt.plot(E[mask], D[mask])
plt.axvline(0, color='k', linestyle='--')
plt.xlim(-10, 6)
plt.xlabel(r'$E - E_F$ (eV)')
plt.ylabel('DOS (states/eV)')
plt.show()


10. The Density of States (DOS) tells us how electronic states are distributed relative to the Fermi level, but it does not provide the absolute energy of the Fermi level. In periodic DFT calculations, all eigenvalues are referenced to an arbitrary internal zero, determined by the basis set, pseudopotentials, and numerical setup. As a result, absolute energy levels cannot be compared directly between two different materials or even between two separate calculations of the same material using different computational parameters. To obtain meaningful, comparable energy references, we compute the work function or, more precisely, the ionization potential of the slab. This quantity gives the energy of the valence‑band maximum (VBM) relative to the vacuum level, which is the same physical reference for every slab. By anchoring the electronic structure to the vacuum level, we can compare band edges, Fermi levels, and work functions across different systems in a consistent way. To perform work‑function or ionization‑potential calculations, the slab must include a sufficiently thick vacuum region. The electrostatic potential must reach a flat plateau in the vacuum, indicating that the potential is constant and free from interactions with the slab. Once this plateau is identified, we extract the vacuum electrostatic potential and reference the VBM (or Fermi level) to it. QE outputs the electrostatic potential in the file potential.cube, which contains the volumetric data needed to compute the averaged potential along the slab normal, allowing post‑processing tools (such as ASE or custom Python scripts) to compute the work function accurately.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.io import read
import numpy as np
import matplotlib.pyplot as plt
from ase.io.cube import read_cube_data

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

prefix = "c2_qe"
outdir = "."

atoms = read('slab_h_qe_high.traj')
atoms.pbc=True

os.mkdir("c2_qe_slab_h_WF")
os.chdir("c2_qe_slab_h_WF")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF', 'H': 'H.pbe-rrkjus_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4 * 80,
            },
            'electrons': {
                'conv_thr': 1e-6
            }
        },
        kpts=(5, 5, 1)
    )

energy = atoms.get_potential_energy()

pp_input = f"""&INPUTPP
    prefix='{prefix}'
    outdir='{outdir}'
    filplot='potential'
    plot_num=11
/

&PLOT
    nfile=1
    filepp(1)='potential'
    iflag=3
    output_format=6
    fileout='potential.cube'
/
"""

with open("pp.in", "w") as f:
    f.write(pp_input)

print("Created pp.in")

os.system("srun --exclusive --ntasks=1 pp.x < pp.in > pp.out")

# ==========================================================
# Read electrostatic potential from QE pp.x cube file
# ==========================================================

data, atoms = read_cube_data("potential.cube")

# QE usually writes potentials in Ry
RY_TO_EV = 13.605693

# convert to eV
data *= RY_TO_EV

# planar average
V_z = data.mean(axis=(0, 1))

# z coordinate
z = np.linspace(
    0,
    atoms.cell[2, 2],
    data.shape[2]
)

# vacuum level
V_vac = np.max(V_z)

# ==========================================================
# Read VBM from QE output
# ==========================================================

VBM = None

with open("espresso.pwo") as f:
    for line in f:
        if "highest occupied level" in line.lower():
            VBM = float(line.split()[-1])
            break

if VBM is None:
    raise RuntimeError(
        "Could not find 'highest occupied level' in espresso.pwo"
    )

# ==========================================================
# Ionization potential
# ==========================================================

IP = V_vac - VBM

print()
print("===================================")
print(f"Vacuum level : {V_vac:10.6f} eV")
print(f"VBM          : {VBM:10.6f} eV")
print("-----------------------------------")
print(f"IP = Evac-VBM: {IP:10.6f} eV")
print("===================================")

# ==========================================================
# Plot
# ==========================================================

plt.figure(figsize=(8,5))

plt.plot(z, V_z, lw=2)

plt.axhline(
    V_vac,
    linestyle='--',
    label=f'Vacuum level = {V_vac:.2f} eV'
)

plt.axhline(
    VBM,
    linestyle='--',
    label=f'VBM = {VBM:.2f} eV'
)

plt.xlabel("z (Å)")
plt.ylabel("Electrostatic potential (eV)")
plt.title("Planar averaged electrostatic potential")
plt.legend()

plt.tight_layout()
plt.show()


os.chdir("../")



The ionization potential of the hydrogen‑passivated diamond (100) slab is calculated to be 2.968856 eV, whereas the pristine diamond (100) surface exhibits a much higher value of 7.172100 eV. This dramatic difference is well‑known experimentally and highlights the crucial role of surface reconstruction and surface termination in determining electronic properties.

Hydrogen termination fundamentally alters the surface electronic environment. By saturating the dangling bonds on the topmost carbon atoms, hydrogen atoms create a surface dipole layer that shifts the electrostatic potential downward. This dipole makes it energetically easier to remove an electron from the surface, resulting in a much lower ionization potential. In practical terms, the H‑terminated surface becomes far more favorable for electron emission, a property exploited in applications such as negative‑electron‑affinity (NEA) diamond devices.

In contrast, the pristine diamond (100) surface lacks these stabilizing dipoles. Its unsaturated dangling bonds produce mid‑gap states and a higher surface potential, making electron extraction significantly more difficult. The high ionization potential reflects this unfavorable electronic environment.

These results demonstrate how surface chemistry directly controls the electronic structure, and they emphasize the importance of accurate surface modeling when studying real materials or designing diamond‑based electronic and optoelectronic devices.

In [ ]:
from ase import Atoms
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.io import read
import numpy as np
import matplotlib.pyplot as plt
from ase.io.cube import read_cube_data


profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

prefix = "c2_qe"
outdir = "."

atoms = read('slab_qe.traj')
atoms.pbc=True

os.mkdir("c2_qe_slab_WF")
os.chdir("c2_qe_slab_WF")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
            },
            'electrons': {
                'conv_thr': 1e-6,
                'electron_maxstep': 300
            }
        },
        kpts=(5, 5, 1)
    )

energy = atoms.get_potential_energy()

pp_input = f"""&INPUTPP
    prefix='{prefix}'
    outdir='{outdir}'
    filplot='potential'
    plot_num=11
/

&PLOT
    nfile=1
    filepp(1)='potential'
    iflag=3
    output_format=6
    fileout='potential.cube'
/
"""

with open("pp.in", "w") as f:
    f.write(pp_input)

print("Created pp.in")

os.system("srun --exclusive --ntasks=1 pp.x < pp.in > pp.out")

# ==========================================================
# Read electrostatic potential from QE pp.x cube file
# ==========================================================

data, atoms = read_cube_data("potential.cube")

# QE usually writes potentials in Ry
RY_TO_EV = 13.605693

# convert to eV
data *= RY_TO_EV

# planar average
V_z = data.mean(axis=(0, 1))

# z coordinate
z = np.linspace(
    0,
    atoms.cell[2, 2],
    data.shape[2]
)

# vacuum level
V_vac = np.max(V_z)

# ==========================================================
# Read VBM from QE output
# ==========================================================

VBM = None

with open("espresso.pwo") as f:
    for line in f:
        if "highest occupied level" in line.lower():
            VBM = float(line.split()[-1])
            break

if VBM is None:
    raise RuntimeError(
        "Could not find 'highest occupied level' in espresso.pwo"
    )

# ==========================================================
# Ionization potential
# ==========================================================

IP = V_vac - VBM

print()
print("===================================")
print(f"Vacuum level : {V_vac:10.6f} eV")
print(f"VBM          : {VBM:10.6f} eV")
print("-----------------------------------")
print(f"IP = Evac-VBM: {IP:10.6f} eV")
print("===================================")

# ==========================================================
# Plot
# ==========================================================

plt.figure(figsize=(8,5))

plt.plot(z, V_z, lw=2)

plt.axhline(
    V_vac,
    linestyle='--',
    label=f'Vacuum level = {V_vac:.2f} eV'
)

plt.axhline(
    VBM,
    linestyle='--',
    label=f'VBM = {VBM:.2f} eV'
)

plt.xlabel("z (Å)")
plt.ylabel("Electrostatic potential (eV)")
plt.title("Planar averaged electrostatic potential")
plt.legend()

plt.tight_layout()
plt.show()


os.chdir("../")



In this tutorial we compared the performance and results of SIESTA and Quantum ESPRESSO for bulk and slab calculations. Despite their methodological differences—LCAO basis sets versus plane waves—both approaches produced consistent and physically meaningful results, including similar lattice parameters, electronic structures, and work‑function trends. SIESTA exhibited occasional SCF convergence difficulties for certain geometries, particularly highly distorted slabs, but once stabilized it remained reliable. In terms of computational efficiency, SIESTA was typically an order of magnitude faster for slab systems, making it attractive for large‑scale or exploratory studies. QE, on the other hand, offers systematic convergence, robust plane‑wave accuracy, and excellent reproducibility, which can be crucial for high‑precision electronic‑structure work. Ultimately, the choice between the two methods depends on the system size, the desired level of accuracy, and the available computational resources, with both codes providing complementary strengths for modern materials‑science workflows.

11. NEB with QE

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.espresso import Espresso, EspressoProfile
import os
import numpy as np
import matplotlib.pyplot as plt

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

os.mkdir("gr_qe_kpts")
os.chdir("gr_qe_kpts")

# Build graphene with total 8 Å vacuum along z
atoms = graphene(a=2.46, vacuum=8.0)
# Optional: center the sheet exactly in the cell along z
atoms.center(axis=2)

kk_array = []
energy_array = []

for kk in range(1, 15, 1): # loop over k-points from 1x1x1 to 8x8x8
 
    atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe'
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
            },
            'electrons': {
                'conv_thr': 1e-8,
                'electron_maxstep': 200
            }
        },
        kpts=(kk, kk, 1)
    )
    energy = atoms.get_total_energy()
    print("k-points:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)

# Plot k-point energy convergence

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs k-points')  # Add a title to the plot
plt.xlabel('k-points')  # Label for the x-axis
plt.ylabel('Energy')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")

view(atoms, viewer='x3d')

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.espresso import Espresso, EspressoProfile
import os
import numpy as np
import matplotlib.pyplot as plt

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

os.mkdir("gr_qe_cutoff")
os.chdir("gr_qe_cutoff")

# Build graphene with total 8 Å vacuum along z
atoms = graphene(a=2.46, vacuum=8.0)
# Optional: center the sheet exactly in the cell along z
atoms.center(axis=2)

kk_array = []
energy_array = []

for kk in range(30, 91, 5): 
 
    atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe'
            },
            'system': {
                'ecutwfc': kk,
                'ecutrho': 4*kk,
            },
            'electrons': {
                'conv_thr': 1e-8,
                'electron_maxstep': 200
            }
        },
        kpts=(12, 12, 1)
    )
    energy = atoms.get_total_energy()
    print("cutoff:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)

# Plot cutoff energy convergence

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs cutoff')  # Add a title to the plot
plt.xlabel('cutoff')  # Label for the x-axis
plt.ylabel('Energy')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")

view(atoms, viewer='x3d')

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.optimize.bfgs import BFGS
from ase import Atom
from ase.io import write

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

# Build graphene with total 8 Å vacuum along z
atoms = graphene(a=2.46, vacuum=8.0)
# Optional: center the sheet exactly in the cell along z
atoms.center(axis=2)

atoms = atoms.repeat((3,3,1))
atoms.append(Atom('Ni', (2.5, 1.5, 10.0)))

os.mkdir("gr1_qe_opt")
os.chdir("gr1_qe_opt")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF', 'Ni': 'ni_pbe_v1.4.uspp.F.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True, 
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
                'occupations': 'smearing',
                'smearing': 'mv',
                'degauss': 0.02,
                # no nspin, no starting_magnetization → non‑polarized
                'assume_isolated': '2D',

                
            },
            'electrons': {
                'conv_thr': 1e-6,
                'electron_maxstep': 500,
                'mixing_mode': 'local-TF',
                'mixing_beta': 0.2,
                'mixing_ndim': 8,
            }
        },
        kpts=(3, 3, 1)
    )

energy = atoms.get_total_energy()

opt = BFGS(atoms, trajectory='atoms.traj')
opt.run(fmax=0.03)

os.chdir("../")

write('gr1_qe.traj', atoms)

view(atoms, viewer='x3d')

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.espresso import Espresso, EspressoProfile
import os
from ase.optimize.bfgs import BFGS
from ase import Atom
from ase.io import write
from ase.io import read

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

atoms = read('gr1_qe.traj')
mg_index = [i for i, a in enumerate(atoms) if a.symbol == 'Ni'][0]
dx = 1.4
dy = 2.2

atoms[mg_index].position += [dx, dy, 0.0]

os.mkdir("gr2_qe_opt")
os.chdir("gr2_qe_opt")

atoms.calc = Espresso(
        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF', 'Ni': 'ni_pbe_v1.4.uspp.F.UPF'},
        profile=profile,
        input_data={
            'control': {
                'calculation': 'scf',
                'prefix': 'c2_qe',
                'tstress': True,
                'tprnfor': True, 
            },
            'system': {
                'ecutwfc': 80,
                'ecutrho': 4*80,
                'occupations': 'smearing',
                'smearing': 'mv',
                'degauss': 0.02,
                # no nspin, no starting_magnetization → non‑polarized
                'assume_isolated': '2D',

                
            },
            'electrons': {
                'conv_thr': 1e-6,
                'electron_maxstep': 700,
                'mixing_mode': 'local-TF',
                'mixing_beta': 0.1,
                'mixing_ndim': 8,
            }
        },
        kpts=(3, 3, 1)
    )

energy = atoms.get_total_energy()

opt = BFGS(atoms, trajectory='atoms.traj')
opt.run(fmax=0.03)

os.chdir("../")

write('gr2_qe.traj', atoms)

view(atoms, viewer='x3d')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from ase.io import read, write
from ase.mep import NEB
from ase.optimize import BFGS
from ase.visualize import view
from ase.calculators.espresso import Espresso, EspressoProfile

profile = EspressoProfile(
    command="srun --exclusive --ntasks=10 --cpus-per-task=1 pw.x",
    pseudo_dir=os.environ["QE_PSEUDO_DIR"],
)

# ------------------------------------------------------------
# 1. Read initial and final geometries
# ------------------------------------------------------------
initial = read('gr1_qe.traj')   # or initial.xyz / initial.traj
final   = read('gr2_qe.traj')

os.mkdir("neb_qe")
os.chdir("neb_qe")

# ------------------------------------------------------------
# 2. Create NEB images (initial + 3 intermediates + final)
# ------------------------------------------------------------
n_images = 5
images = [initial]

for i in range(n_images - 2):
    images.append(initial.copy())

images.append(final)

neb = NEB(images)
neb.interpolate()   # linear interpolation

# ------------------------------------------------------------
# 3. Attach a *separate* QE calculator to each image
# ------------------------------------------------------------
def make_qe_calc(label):
    return Espresso(
                        pseudopotentials={'C': 'C.pbe-n-kjpaw_psl.1.0.0.UPF', 'Ni': 'ni_pbe_v1.4.uspp.F.UPF'},
                        profile=profile,
                        input_data={
                            'control': {
                                'calculation': 'scf',
                                'prefix': 'c2_qe',
                                'tstress': True,
                                'tprnfor': True, 
                            },
                            'system': {
                                'ecutwfc': 80,
                                'ecutrho': 4*80,
                                'occupations': 'smearing',
                                'smearing': 'mv',
                                'degauss': 0.02,
                                # no nspin, no starting_magnetization → non‑polarized
                                'assume_isolated': '2D',

                                
                            },
                            'electrons': {
                                'conv_thr': 1e-6,
                                'electron_maxstep': 1000,
                                'mixing_mode': 'local-TF',
                                'mixing_beta': 0.1,
                                'mixing_ndim': 8,
                            }
                        },
                        kpts=(3, 3, 1)
                    )   

for i, img in enumerate(images):
    img.calc = make_qe_calc(f'neb_img_{i}')

# ------------------------------------------------------------
# 4. Optimize the NEB path
# ------------------------------------------------------------
opt = BFGS(neb, logfile='neb.log')
opt.run(fmax=0.05)  # convergence criterion for forces on images

# ------------------------------------------------------------
# 5. Collect energies and compute relative profile
# ------------------------------------------------------------
energies = [img.get_potential_energy() for img in images]
E0 = energies[0]
rel_energies = [E - E0 for E in energies]

# Identify TS (highest energy image)
ts_index = int(np.argmax(energies))
ts_image = images[ts_index]
write('TS.traj', ts_image)

# ------------------------------------------------------------
# 6. Plot NEB energy profile
# ------------------------------------------------------------
plt.figure()
plt.plot(range(n_images), rel_energies, '-o')
plt.xlabel('Image index')
plt.ylabel('Relative energy (eV)')
plt.title('NEB Energy Profile')
plt.grid(True)
plt.tight_layout()
plt.savefig('neb_profile.png', dpi=200)

print("Absolute energies (eV):", energies)
print("TS image index:", ts_index)
print("TS geometry saved as TS.traj")
os.chdir("../")
view(ts_image, viewer='x3d')